# 06 Scenario study: paired comparison vs baseline

Ticket T5.2.1. **Simulated, benchmark-parameterized line; not plant data.** Baseline v1.0 (config hash `2f9b035c9964`, tag `baseline-v1.0`).

**What was pre-registered.** Every scenario, level, metric and decision rule below was fixed in `docs/scenario_matrix.md` and `docs/domain/scenario_hypotheses.md`, committed in `bc2518f` before any scenario run. The **main set** is the 10 comparisons SC-01 to SC-04; Holm adjusts their p-values on the primary metric only. **SC-05 is exploratory** and is reported unadjusted. Secondary and co-primary metrics are reported unadjusted as supporting evidence.

**Amendment E1.** The first run exposed a model defect with parallel AVI units (see `docs/scenario_matrix.md`, E1). It was fixed and everything re-run; only sc01 and sc05 changed.

**Method** (`src/ffdt/experiments/stats.py`): replication *r* of each scenario is paired with replication *r* of the baseline (common random numbers). On the 30 differences: mean, t-based 95% CI, paired t-test, percentile bootstrap 95% CI (10,000 resamples) as a cross-check, and the unpaired Welch CI width for comparison. Practical significance (TARGET): at least +3% good vials per PPT hour, or 2 OEE points.

In [1]:
import pandas as pd
import plotly.graph_objects as go

from ffdt import viz
from ffdt.experiments import stats as S

df = S.run()  # reads results/<scenario>/*.parquet, writes results/comparison.{csv,parquet}
pd.set_option("display.width", 200)
prim = df[df.primary].set_index("scenario")
prim[
    ["set", "base_mean", "mean_diff", "pct_diff", "ci_low", "ci_high", "p", "p_holm", "practical"]
].round(4)

,set,base_mean,mean_diff,pct_diff,ci_low,ci_high,p,p_holm,practical
scenario,,,,,,,,,
sc01_avi_plus1,main,8252.2042,20.2844,0.0025,-12.2439,52.8127,0.2123,0.8492,False
sc02_batch_x0p5,main,8252.2042,-1396.4690,-0.1692,-1509.2515,-1283.6866,0.0000,0.0000,True
sc02_batch_x2,main,8252.2042,487.2035,0.0590,359.8361,614.5709,0.0000,0.0000,True
sc03_inspector_minus1,main,8252.2042,0.1145,0.0000,-0.4269,0.6558,0.6686,0.9221,False
sc03_inspector_plus1,main,8252.2042,0.0010,0.0000,-0.0010,0.0030,0.3256,0.9221,False
sc03_mechanic_plus1,main,8252.2042,4.1597,0.0005,-4.0283,12.3477,0.3074,0.9221,False
sc03_operator_minus1,main,8252.2042,-432.2949,-0.0524,-463.1454,-401.4445,0.0000,0.0000,True
sc03_operator_plus1,main,8252.2042,291.7417,0.0354,251.6216,331.8618,0.0000,0.0000,True
sc04_changeover_m25,main,8252.2042,914.7144,0.1108,889.1915,940.2374,0.0000,0.0000,True


## Primary metric: good vials per scheduled-production hour (RESULT, simulated)

In [2]:
BLUE, ORANGE, MUTED = viz.series_color(), "#eb6834", "#52514e"
p = prim.iloc[::-1]
base = p.base_mean.iloc[0]
labels = [
    f"{n} (exploratory)" if s == "exploratory" else n for n, s in zip(p.index, p.set, strict=True)
]
fig = go.Figure(
    go.Scatter(
        x=100 * p.pct_diff,
        y=labels,
        mode="markers",
        marker=dict(size=10, color=BLUE, line=dict(width=2, color="#fcfcfb")),
        error_x=dict(
            type="data",
            symmetric=False,
            thickness=2,
            width=0,
            color=BLUE,
            array=100 * (p.ci_high - p.mean_diff) / base,
            arrayminus=100 * (p.mean_diff - p.ci_low) / base,
        ),
        customdata=p[["mean_diff", "ci_low", "ci_high", "p_holm"]].values,
        hovertemplate="%{y}<br>%{x:.1f}% (%{customdata[0]:.0f} vials/h)<br>"
        "95% CI %{customdata[1]:.0f} to %{customdata[2]:.0f}<br>Holm p %{customdata[3]:.2g}<extra></extra>",
    )
)
fig.add_vline(x=0, line=dict(color=MUTED, width=1))
for x in (-3, 3):
    fig.add_vline(x=x, line=dict(color=MUTED, width=1, dash="dash"))
fig.add_annotation(
    x=3,
    y=1.0,
    yref="paper",
    text="±3% practical threshold (TARGET)",
    showarrow=False,
    xanchor="left",
    font=dict(size=11, color=MUTED),
)
viz._layout(fig, "Change in good vials per scheduled-production hour vs baseline", 460)
fig.update_xaxes(title="% vs baseline (paired 95% CI, 30 replications)", ticksuffix="%")
fig.update_layout(template="plotly_white", plot_bgcolor="#fcfcfb", paper_bgcolor="#fcfcfb")
fig.write_image("../reports/figures/06_scenario_forest.png", scale=2, width=820)
fig

## Common random numbers: paired vs unpaired 95% CI half-width (primary metric)

The unpaired width is what the same 30 replications would give if scenario and baseline used independent random streams.

For scenarios that change the line (SC-01, SC-02, operators, SC-04, SC-05), pairing narrows the CI 1.3x to 7x. For ±1 inspector and +1 mechanic the lever almost never binds, so the paired difference is ~0 by construction and the ratio is not meaningful.

In [3]:
w = prim[["paired_half_width", "unpaired_half_width"]].copy()
w["reduction_x"] = w.unpaired_half_width / w.paired_half_width
display(w.round(2))
q = w.iloc[::-1]
fig = go.Figure(
    [
        go.Bar(
            y=q.index,
            x=q.unpaired_half_width,
            orientation="h",
            name="Unpaired (independent streams)",
            marker=dict(color=ORANGE, cornerradius=4),
            hovertemplate="%{y}: ±%{x:.1f} vials/h<extra>unpaired</extra>",
        ),
        go.Bar(
            y=q.index,
            x=q.paired_half_width,
            orientation="h",
            name="Paired (common random numbers)",
            marker=dict(color=BLUE, cornerradius=4),
            hovertemplate="%{y}: ±%{x:.1f} vials/h<extra>paired</extra>",
        ),
    ]
)
viz._layout(fig, "95% CI half-width on good vials per PPT hour: paired vs unpaired", 480)
fig.update_layout(
    barmode="group",
    bargroupgap=0.1,
    template="plotly_white",
    plot_bgcolor="#fcfcfb",
    paper_bgcolor="#fcfcfb",
    legend=dict(orientation="h", y=-0.15, x=0),
)
fig.update_xaxes(title="half-width, vials per PPT hour")
fig.write_image("../reports/figures/06_crn_halfwidth.png", scale=2, width=820)
fig

,paired_half_width,unpaired_half_width,reduction_x
scenario,,,
sc01_avi_plus1,32.53,169.60,5.21
sc02_batch_x0p5,112.78,144.36,1.28
sc02_batch_x2,127.37,196.46,1.54
sc03_inspector_minus1,0.54,173.82,321.06
sc03_inspector_plus1,0.00,173.83,87483.52
sc03_mechanic_plus1,8.19,173.62,21.20
sc03_operator_minus1,30.85,166.45,5.40
sc03_operator_plus1,40.12,178.38,4.45
sc04_changeover_m25,25.52,182.26,7.14


## Bootstrap cross-check

Stated tolerance (`BOOT_TOL = 0.15`): each bootstrap CI end within 15% of the paired t half-width of the matching t-CI end. Rows that miss it are listed with the share of replications whose difference is exactly 0.

In [4]:
print(
    f"bootstrap agrees on {df.boot_agrees.sum()} of {len(df)} scenario x metric rows; "
    f"primary metric: {prim.boot_agrees.sum()} of {len(prim)}"
)
df.loc[
    ~df.boot_agrees,
    [
        "scenario",
        "metric",
        "mean_diff",
        "ci_low",
        "ci_high",
        "boot_low",
        "boot_high",
        "zero_diff_frac",
    ],
].round(4)

bootstrap agrees on 54 of 77 scenario x metric rows; primary metric: 9 of 11


,scenario,metric,mean_diff,ci_low,ci_high,boot_low,boot_high,zero_diff_frac
11,sc02_batch_x0p5,wait_staff_share,-0.0003,-0.0008,0.0001,-0.0009,0.0000,0.9333
18,sc02_batch_x2,wait_staff_share,0.0001,-0.0001,0.0004,0.0000,0.0004,0.9333
19,sc02_batch_x2,mean_lead_time_h,8.4718,2.9354,14.0082,3.8428,14.0751,0.0000
25,sc03_inspector_minus1,wait_staff_share,0.0000,-0.0000,0.0000,-0.0000,0.0000,0.9333
26,sc03_inspector_minus1,mean_lead_time_h,0.4172,-0.0117,0.8461,0.0783,0.8734,0.0333
28,sc03_inspector_plus1,good_per_ppt_hour,0.0010,-0.0010,0.0030,-0.0000,0.0029,0.9333
29,sc03_inspector_plus1,line_oee,0.0000,-0.0000,0.0000,-0.0000,0.0000,0.9333
30,sc03_inspector_plus1,line_A,-0.0000,-0.0000,0.0000,-0.0000,0.0000,0.9333
31,sc03_inspector_plus1,changeover_share,0.0000,-0.0000,0.0000,-0.0000,0.0000,0.9333
34,sc03_inspector_plus1,makespan_h,-0.0000,-0.0000,0.0000,-0.0000,0.0000,0.9333


**Reading the disagreements.** Most come from degenerate differences: the lever never bound in most replications (difference exactly 0 in 28 of 30) and one or two replications carry an outlier, so the t-interval's normality assumption fails. Example: +1 mechanic changes only 2 of 30 replications. There the bootstrap interval is the more honest one. Two rows are not degenerate and are skewed instead: lead time for `sc02_batch_x2` (both intervals exclude 0, same conclusion) and lead time for `sc03_inspector_minus1`, where the t-CI includes 0 and the bootstrap CI excludes it. That secondary result is therefore reported as **inconclusive** (+0.4 h on an 89 h baseline either way).

**On the primary metric** both intervals reach the same conclusion in every scenario: no detectable effect for +1 mechanic and ±1 inspector, a clear effect everywhere else.

## Family co-primary and secondary metrics (unadjusted)

In [5]:
sec = df[
    df.metric.isin(
        ["changeover_share", "wait_staff_share", "line_oee", "mean_lead_time_h", "makespan_h"]
    )
]
sec.pivot_table(index="scenario", columns="metric", values="mean_diff").round(4)

metric,changeover_share,line_oee,makespan_h,mean_lead_time_h,wait_staff_share
scenario,,,,,
sc01_avi_plus1,0.0006,0.0014,-0.1568,-0.0024,0.0000
sc02_batch_x0p5,0.0655,-0.0970,33.3294,27.3421,-0.0003
sc02_batch_x2,-0.0304,0.0338,-30.7403,8.4718,0.0001
sc03_inspector_minus1,0.0000,0.0000,-0.0011,0.4172,0.0000
sc03_inspector_plus1,0.0000,0.0000,-0.0000,0.0000,0.0000
sc03_mechanic_plus1,0.0001,0.0003,-0.0371,-0.0054,-0.0022
sc03_operator_minus1,-0.0143,-0.0300,14.1397,4.1329,-0.0001
sc03_operator_plus1,0.0097,0.0203,-21.4755,-3.1236,0.0001
sc04_changeover_m25,-0.0456,0.0635,-51.2498,-4.3931,0.0002


## Reading the results (no hypothesis verdicts here)

Hypothesis verdicts need the baseline constraint (T5.2.3: three bottleneck methods) and are written in the scenario report, not here. This notebook only establishes which effects are statistically and practically distinguishable from zero. All numbers are RESULTS from `results/comparison.csv`; the public copy will be `reports/results_final.json` (later ticket).